In [1]:
!wget -nc https://storage.googleapis.com/tensorflow-1-public/course3/imdb_vocab_subwords.txt

--2026-05-03 15:07:44--  https://storage.googleapis.com/tensorflow-1-public/course3/imdb_vocab_subwords.txt
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.141.207, 142.251.2.207, 74.125.137.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.141.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53151 (52K) [text/plain]
Saving to: ‘imdb_vocab_subwords.txt’

imdb_vocab_subwords 100%[===================>]  51.91K  --.-KB/s    in 0.001s  

2026-05-03 15:07:44 (71.0 MB/s) - ‘imdb_vocab_subwords.txt’ saved [53151/53151]



In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import keras_nlp

In [3]:
imdb = tfds.load('imdb_reviews', as_supervised=True, data_dir="./data")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling data/imdb_reviews/plain_text/incomplete.G8E08T_1.0.0/imdb_reviews-train.tfrecord*...:   0%|         …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling data/imdb_reviews/plain_text/incomplete.G8E08T_1.0.0/imdb_reviews-test.tfrecord*...:   0%|          …

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling data/imdb_reviews/plain_text/incomplete.G8E08T_1.0.0/imdb_reviews-unsupervised.tfrecord*...:   0%|  …

Dataset imdb_reviews downloaded and prepared to data/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [4]:
train_reviews = imdb['train'].map(lambda review, label: review)
test_reviews = imdb['test'].map(lambda review, label : review)

train_labels = imdb['train'].map(lambda review, label : label)
test_labels = imdb['test'].map(lambda review, label : label)

In [5]:
subword_tokenizer = keras_nlp.tokenizers.WordPieceTokenizer(vocabulary="./imdb_vocab_subwords.txt")

In [6]:
SHUFFLE_BUFFER_SIZE = 10000
PREFETCH_BUFFER_SIZE = tf.data.AUTOTUNE
BATCH_SIZE = 256
PADDING_TYPE = "pre"
TRUNC_TYPE = "post"

In [7]:
def padding_function(sequences):
  sequences = sequences.ragged_batch(batch_size = sequences.cardinality())
  sequences = sequences.get_single_element()
  padded_sequences = tf.keras.utils.pad_sequences(sequences.numpy(), padding=PADDING_TYPE, truncating=TRUNC_TYPE)
  padded_sequences = tf.data.Dataset.from_tensor_slices(padded_sequences)
  return padded_sequences

In [8]:
train_sequences_subword = train_reviews.map(lambda review: subword_tokenizer.tokenize(review)).apply(padding_function)

In [9]:
test_sequence_subword = test_reviews.map(lambda review: subword_tokenizer.tokenize(review)).apply(padding_function)

In [10]:
train_data_vect = tf.data.Dataset.zip(train_sequences_subword, train_labels)
test_data_vect = tf.data.Dataset.zip(test_sequence_subword, test_labels)

In [11]:
test_data_vect = test_data_vect.batch(BATCH_SIZE).prefetch(PREFETCH_BUFFER_SIZE).cache()
train_data_vect = train_data_vect.shuffle(SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE).prefetch(PREFETCH_BUFFER_SIZE).cache()

In [12]:
EMBEDDING_DIM = 64
LSTM_DIM = 64
DENSE_DIM = 64

model = tf.keras.Sequential([
    tf.keras.Input(shape=(None,)),
    tf.keras.layers.Embedding(input_dim=subword_tokenizer.vocabulary_size(), output_dim=EMBEDDING_DIM),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(LSTM_DIM)),
    tf.keras.layers.Dense(DENSE_DIM, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 64)       │       488,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 563,009 (2.15 MB)

 Trainable params: 563,009 (2.15 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [15]:
epoches = 5
history = model.fit(train_data_vect, epochs=epoches, validation_data=test_data_vect)

Epoch 1/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 56s 574ms/step - accuracy: 0.8800 - loss: 0.3004 - val_accuracy: 0.8539 - val_loss: 0.3552
Epoch 2/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 55s 559ms/step - accuracy: 0.9219 - loss: 0.2079 - val_accuracy: 0.8580 - val_loss: 0.3751
Epoch 3/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 55s 559ms/step - accuracy: 0.9393 - loss: 0.1693 - val_accuracy: 0.8574 - val_loss: 0.4149
Epoch 4/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 82s 560ms/step - accuracy: 0.9296 - loss: 0.1893 - val_accuracy: 0.8374 - val_loss: 0.4287
Epoch 5/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 62s 639ms/step - accuracy: 0.9352 - loss: 0.1752 - val_accuracy: 0.8377 - val_loss: 0.5440


In [ ]:
def plot_loss_acc(history):
  '''Plots the training and validation loss and accuracy from a history object'''
  acc = history.history['accuracy']
  val_acc = history.history['val_accuracy']
  loss = history.history['loss']
  val_loss = history.history['val_loss']

  epochs = range(len(acc))

  fig, ax = plt.subplots(1,2, figsize=(12, 6))
  ax[0].plot(epochs, acc, 'bo', label='Training accuracy')
  ax[0].plot(epochs, val_acc, 'b', label='Validation accuracy')
  ax[0].set_title('Training and validation accuracy')
  ax[0].set_xlabel('epochs')
  ax[0].set_ylabel('accuracy')
  ax[0].legend()

  ax[1].plot(epochs, loss, 'bo', label='Training Loss')
  ax[1].plot(epochs, val_loss, 'b', label='Validation Loss')
  ax[1].set_title('Training and validation loss')
  ax[1].set_xlabel('epochs')
  ax[1].set_ylabel('loss')
  ax[1].legend()

  plt.show()

plot_loss_acc(history)

In [18]:
# Shutdown the kernel to free up resources.
# Note: You can expect a pop-up when you run this cell. You can safely ignore that and just press `Ok`.

from IPython import get_ipython

k = get_ipython().kernel

k.do_shutdown(restart=False)

{'status': 'ok', 'restart': False}